In [1]:
import csv
from collections import defaultdict
import pandas

Count rows

In [2]:
with open ("data/posts.csv", "r", newline="", encoding="utf-8") as data:
    reader = csv.DictReader(data)
    kw_counts = defaultdict(int)
    for row in reader:
        kw_counts[row['keyword']] += 1
        
total = 0
for key in kw_counts:
    if kw_counts[key] != 0:
        total += kw_counts[key]


print (kw_counts)
print(total)

defaultdict(<class 'int'>, {'twitter': 2057, 'elon': 2009, 'twitter refugee': 2007, 'bluesky vs twitter': 2009, 'twitter is dead': 2016, 'quit twitter': 2008, 'miss twitter': 2059})
14165


Stop being a chud and start using pandas.

In [3]:
df = pandas.read_csv("data/posts.csv")
print(f"Total posts: {len(df)}")
print(f"Duplicates: {df['uri'].duplicated().sum()}")
print(f"Is reply True: {df['is_reply'].sum()}")
print(f"Is reply False: {(df['is_reply'] == False).sum()}")
print(f"\nReply count stats:")
print(df['reply_count'].describe())

Total posts: 14165
Duplicates: 120
Is reply True: 6623
Is reply False: 7542

Reply count stats:
count    14165.000000
mean         1.115708
std         17.730914
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max       1996.000000
Name: reply_count, dtype: float64


Get rid of dupes (based on the uri of the post) - better than checking for dupes on text, since multiple users can post the same text.

In [4]:
print(f"Prior to removing dupes: {len(df)}")
df = df.drop_duplicates(subset="uri")
print(f"After removing dupes: {len(df)}")
df.to_csv("data/posts_no_dupes.csv", index = False)

Prior to removing dupes: 14165
After removing dupes: 14045


See the top-10 posts with most replies.

In [12]:
sorted_df = df.sort_values(by="reply_count", ascending=False)

top_replied_posts_df = sorted_df[["uri","text","reply_count", "keyword"]].head(10)
top_replied_posts_df.to_csv("data/top_replied_posts.csv", index = False)
uris_all = [uri for uri in top_replied_posts_df["uri"]]
print(uris_all)


['at://did:plc:xyddpg6usmgh2t2jgf4e37yk/app.bsky.feed.post/3mhvfinvcgc2q', 'at://did:plc:ul5n745uxwymqppvpiwtpoa5/app.bsky.feed.post/3mkej7zyfnk2h', 'at://did:plc:5mqpgxjffcckasqv7h6g7itu/app.bsky.feed.post/3m3crrs6ark2y', 'at://did:plc:tsf3qulwq25yo27j6eznkfix/app.bsky.feed.post/3mc2crp2ahc27', 'at://did:plc:4q4ziw4qk2hxxaxfu7jweuey/app.bsky.feed.post/3mmak4a4rns2k', 'at://did:plc:ji7lroxun3yvv2pxhcf7jqsn/app.bsky.feed.post/3mmagsxpukq22', 'at://did:plc:37ukqjgnt2puqbdvxo6jw4le/app.bsky.feed.post/3mmafhs4fws2z', 'at://did:plc:t4i3a4fawuzge3dsw2i2h2fw/app.bsky.feed.post/3mbhvqutprk2a', 'at://did:plc:t46sqvutibvsmjgwn6r6izve/app.bsky.feed.post/3ltfs7bk3js2i', 'at://did:plc:dy4mk6ej5d7hlqgfeqjft3hd/app.bsky.feed.post/3mcnhpj3qj22j']


In [13]:
top_replied_posts_twtvsbsky = sorted_df[sorted_df['keyword'] == 'bluesky vs twitter'][["uri",'text', 'reply_count', 'keyword']].head(10)
print(top_replied_posts_twtvsbsky)
uris_twtvsbsky = [uri for uri in top_replied_posts_twtvsbsky["uri"]]

                                                    uri  \
6935  at://did:plc:t46sqvutibvsmjgwn6r6izve/app.bsky...   
6322  at://did:plc:3iwge6tzr76tkt6xdwyfs6mr/app.bsky...   
7333  at://did:plc:jimtocu7irkkkupvh7g34rhs/app.bsky...   
6394  at://did:plc:rkfwqt5jedajtdnkx5kvedfo/app.bsky...   
6459  at://did:plc:4vrssqepg6uj4tj5us7tnfgt/app.bsky...   
6999  at://did:plc:psxf6wrijwkudvi2etmxsess/app.bsky...   
7989  at://did:plc:sgti3jsgu3luif24tokvth3a/app.bsky...   
6708  at://did:plc:kyphkmluigfakab42kfn5ri2/app.bsky...   
6916  at://did:plc:cak4klqoj3bqgk5rj6b4f5do/app.bsky...   
7051  at://did:plc:slwpvr5uwq7dqv4nur35dlji/app.bsky...   

                                                   text  reply_count  \
6935  Twitter vs Threads vs BlueSky \n\nHave definit...           83   
6322  i was wondering if we could maybe argue about ...           56   
7333  I've been keeping a closer eye on traffic from...           41   
6394  The playoffs are alive on BlueSky, the officia...       

Get all post uris that we want to fetch threads for.

In [16]:
uris = uris_all + uris_twtvsbsky
print(uris)
print(len(uris))

['at://did:plc:xyddpg6usmgh2t2jgf4e37yk/app.bsky.feed.post/3mhvfinvcgc2q', 'at://did:plc:ul5n745uxwymqppvpiwtpoa5/app.bsky.feed.post/3mkej7zyfnk2h', 'at://did:plc:5mqpgxjffcckasqv7h6g7itu/app.bsky.feed.post/3m3crrs6ark2y', 'at://did:plc:tsf3qulwq25yo27j6eznkfix/app.bsky.feed.post/3mc2crp2ahc27', 'at://did:plc:4q4ziw4qk2hxxaxfu7jweuey/app.bsky.feed.post/3mmak4a4rns2k', 'at://did:plc:ji7lroxun3yvv2pxhcf7jqsn/app.bsky.feed.post/3mmagsxpukq22', 'at://did:plc:37ukqjgnt2puqbdvxo6jw4le/app.bsky.feed.post/3mmafhs4fws2z', 'at://did:plc:t4i3a4fawuzge3dsw2i2h2fw/app.bsky.feed.post/3mbhvqutprk2a', 'at://did:plc:t46sqvutibvsmjgwn6r6izve/app.bsky.feed.post/3ltfs7bk3js2i', 'at://did:plc:dy4mk6ej5d7hlqgfeqjft3hd/app.bsky.feed.post/3mcnhpj3qj22j', 'at://did:plc:t46sqvutibvsmjgwn6r6izve/app.bsky.feed.post/3ltfs7bk3js2i', 'at://did:plc:3iwge6tzr76tkt6xdwyfs6mr/app.bsky.feed.post/3mf5ta52w7c2p', 'at://did:plc:jimtocu7irkkkupvh7g34rhs/app.bsky.feed.post/3lkvp53pxvk27', 'at://did:plc:rkfwqt5jedajtdnkx5kvedf

In [19]:
edges_df = pandas.read_csv('data/edges.csv')
print(f"Total edges: {len(edges_df)}")
print(f"Unique from_did: {edges_df['from_did'].nunique()}")
print(f"Unique to_did: {edges_df['to_did'].nunique()}")

Total edges: 2825
Unique from_did: 2062
Unique to_did: 503


## **Network Analysis**

In [ ]:
import pandas as pd
import networkx as nx
import community as community_louvain
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

print(edges_df.head())